# 1. Purpose

The purpose of this project is to analyze Los Angeles property listings collected on redfin.com from November 2024, and attempt to create a model to predict the cost of a house given certain criteria.

# 2. Importing Datasets

In [1]:
# loading necessary packages
import sqlite3
import numpy as np
import pandas as pd
import glob
import plotly.express as px


In [2]:
# load properties downloaded from Redfin
csv_files = glob.glob('la_datasets/' + "*.csv")

# initialize list to store dataframes
dfs = []

# loop through each csv file and append to dfs
for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)

# merge all dataframes together
raw_properties = pd.concat(dfs, ignore_index=True)
print(f"There is a total of {len(raw_properties)} properties on sale.")
raw_properties.head()

There is a total of 8654 properties on sale.


,SALE TYPE,SOLD DATE,PROPERTY TYPE,ADDRESS,CITY,STATE OR PROVINCE,ZIP OR POSTAL CODE,PRICE,BEDS,BATHS,...,STATUS,NEXT OPEN HOUSE START TIME,NEXT OPEN HOUSE END TIME,URL (SEE https://www.redfin.com/buy-a-home/comparative-market-analysis FOR INFO ON PRICING),SOURCE,MLS#,FAVORITE,INTERESTED,LATITUDE,LONGITUDE
0,"In accordance with local MLS rules, some MLS l...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MLS Listing,NaN,Multi-Family (2-4 Unit),1239 Gordon St,Los Angeles,CA,90038.0,1995000.0,12.0,6.0,...,Active,NaN,NaN,https://www.redfin.com/CA/Los-Angeles/1239-Gor...,TheMLS,24-461931,N,Y,34.093744,-118.320290
2,MLS Listing,NaN,Multi-Family (5+ Unit),858 N Hudson Ave,Los Angeles,CA,90038.0,3599000.0,12.0,14.0,...,Active,NaN,NaN,https://www.redfin.com/CA/Los-Angeles/858-N-Hu...,TheMLS,24-461337,N,Y,34.086945,-118.331881
3,MLS Listing,NaN,Multi-Family (2-4 Unit),5851 La Mirada Ave,Los Angeles,CA,90038.0,1300000.0,4.0,2.0,...,Active,NaN,NaN,https://www.redfin.com/CA/Los-Angeles/5851-La-...,TheMLS,24-461779,N,Y,34.094047,-118.317588
4,MLS Listing,NaN,Condo/Co-op,945 N Hudson Ave #104,Los Angeles,CA,90038.0,714900.0,2.0,2.0,...,Active,NaN,NaN,https://www.redfin.com/CA/Los-Angeles/945-N-Hu...,CRMLS,RS24229871,N,Y,34.088349,-118.332395


# 3. Cleaning the Dataset

In [3]:
def clean_properties(raw_properties):
    """ cleans Redfin properties dataset by dropping unnecessary columns, removing rows
        with all missing values, removing duplicate rows, standardizing text data by 
        converting to lowercase, and converting numeric values from float to int

    Args:
        raw_properties (pandas DataFrame): the raw Redfin dataset

    Returns:
        properties: a pandas DataFrame of the cleaned data
    """

    properties = raw_properties.copy()

    # dropping unnecessary columns
    properties = properties.drop(columns=["SALE TYPE", "SOLD DATE", "STATE OR PROVINCE", "NEXT OPEN HOUSE START TIME",
                                          "STATUS", "NEXT OPEN HOUSE END TIME","SOURCE", "MLS#", "FAVORITE", "INTERESTED"])
    
    properties = properties.dropna(how='all') # removing rows with all missing values 
    properties = properties.drop_duplicates(ignore_index=True) # removing duplicate rows


    # renaming columns for convenience
    properties.columns = properties.columns.str.lower() # convert all columns to lowercase
    properties.columns = properties.columns.str.replace(" ", "_", regex=True) # replacing space with underscore
    properties = properties.rename(columns={"url_(see_https://www.redfin.com/buy-a-home/comparative-market-analysis_for_info_on_pricing)" : 'url',
                                            "zip_or_postal_code": "zip_code",
                                            "state_or_province": "state",
                                            "$/square_feet": "cost_per_sq_feet",
                                            "hoa/month": "monthly_hoa_cost"
                                            })

    # convert numeric columns from float to int
    numeric_cols = ['zip_code', 'price', 'beds', 'baths', 'square_feet', 'lot_size', 'year_built',
                    'days_on_market', 'cost_per_sq_feet', 'monthly_hoa_cost']
    for col in numeric_cols: properties[col] = properties[col].fillna(0).astype(int) # replaces na with 0

    # remove duplicate rows
    properties = properties.drop_duplicates()

    # remove row where zip code is 0 (only one such row in dataset)
    properties = properties[properties["zip_code"] != 0]

    return properties

# cleaning dataset
properties = raw_properties.copy()
properties = clean_properties(properties)
properties.head(10)

,property_type,address,city,zip_code,price,beds,baths,location,square_feet,lot_size,year_built,days_on_market,cost_per_sq_feet,monthly_hoa_cost,url,latitude,longitude
0,Multi-Family (2-4 Unit),1239 Gordon St,Los Angeles,90038,1995000,12,6,Hollywood,4119,8297,1921,2,484,0,https://www.redfin.com/CA/Los-Angeles/1239-Gor...,34.093744,-118.320290
1,Multi-Family (5+ Unit),858 N Hudson Ave,Los Angeles,90038,3599000,12,14,Hollywood,5153,4470,2024,2,698,0,https://www.redfin.com/CA/Los-Angeles/858-N-Hu...,34.086945,-118.331881
2,Multi-Family (2-4 Unit),5851 La Mirada Ave,Los Angeles,90038,1300000,4,2,Hollywood,2118,7202,1923,2,614,0,https://www.redfin.com/CA/Los-Angeles/5851-La-...,34.094047,-118.317588
3,Condo/Co-op,945 N Hudson Ave #104,Los Angeles,90038,714900,2,2,WLA - West Los Angeles,890,6803,1991,2,803,425,https://www.redfin.com/CA/Los-Angeles/945-N-Hu...,34.088349,-118.332395
4,Condo/Co-op,5806 Waring Ave #5,Los Angeles,90038,898000,2,2,Hollywood,1320,12464,2007,4,680,375,https://www.redfin.com/CA/Los-Angeles/5806-War...,34.085128,-118.324697
5,Multi-Family (2-4 Unit),5832 Camerford Ave,Los Angeles,90038,3599000,4,4,699 - Not Defined,2076,6502,1911,4,1734,0,https://www.redfin.com/CA/Los-Angeles/5832-Cam...,34.084184,-118.325581
6,Condo/Co-op,803 Wilcox Ave #3,Los Angeles,90038,995000,3,2,Hollywood,1510,11450,2008,9,659,730,https://www.redfin.com/CA/Los-Angeles/803-Wilc...,34.085522,-118.331315
7,Single Family Residential,838 N Mccadden Pl,Los Angeles,90038,1588000,3,3,Hollywood,2192,1862,2021,10,724,120,https://www.redfin.com/CA/Los-Angeles/838-N-Mc...,34.086419,-118.337268
8,Single Family Residential,1232-1/8 N Cahuenga Blvd,Los Angeles,90038,715000,2,1,C20 - Hollywood,760,1228,1924,13,941,0,https://www.redfin.com/CA/Los-Angeles/1232-1-8...,34.093515,-118.328610
9,Single Family Residential,6029 Eleanor Ave,Los Angeles,90038,1749999,5,3,Hollywood,1911,4496,1904,16,916,0,https://www.redfin.com/CA/Los-Angeles/6029-Ele...,34.090066,-118.323305


# 4. Exploratory Data Analysis

In [4]:
NUM_OF_PROPERTIES = len(properties)

# count number of houses in each location
location_count = properties.groupby(["location"]).size().reset_index(name = "count").sort_values(by="count", ascending=True)

# evaluate percentage of houses in each location
location_count["percentage"] = (location_count["count"]/NUM_OF_PROPERTIES).round(5)
location_count.head()

,location,count,percentage
88,MR - Marbrisa,1,0.0002
103,OTHR,1,0.0002
38,C11 - Venice,1,0.0002
102,OLYM - Mount Olympus,1,0.0002
109,STUD - Studio City,1,0.0002


As we can see here, Downtown L.A. has the most amount of houses for sale in November 2024 on `Redfin.com`.

In [5]:
# generate bar chart to see percentage of houses from each neighborhood
fig = px.bar(location_count, x = "count", y = "location", 
             hover_name = "location", hover_data = ["percentage", "count"],
             height=2000, color = "percentage", color_continuous_scale = "sunset",
             title = "Percentage of Houses from Each Location",
             labels = {"location": "Location",
                       "count": "Count"}
             )
fig.update_layout(bargap=0.25)
fig.show()

In [6]:
NUM_OF_PROPERTIES = len(properties)

# count number of houses in each zip code
zip_count = properties.groupby(["zip_code"]).size().reset_index(name = "count").sort_values(by="count", ascending=True)

# evaluate percentage of houses in each location
zip_count["percentage"] = (zip_count["count"]/NUM_OF_PROPERTIES).round(5)

# convert zip codes to strings properly display zip codes in bar chart
zip_count["zip_code"] = zip_count["zip_code"].apply(str)

zip_count.head()

,zip_code,count,percentage
74,93543,1,0.0002
72,90402,1,0.0002
71,90301,1,0.0002
68,90280,1,0.0002
66,90241,1,0.0002


In [7]:
# generate bar chart to see percentage of houses from each zip code
fig = px.bar(zip_count, x = "count", y = "zip_code", 
             hover_name = "zip_code", hover_data = ["percentage", "count"],
             height=2000, color = "percentage", color_continuous_scale = "sunset",
             title = "Percentage of Houses from Each Zip Code",
             labels = {"zip_code": "Zip Code",
                       "count": "Count"}
             )
fig.update_layout(bargap=0.25)
fig.show()

In [8]:
# TODO: use sqlite to combine location_count and zip_count
# (i.e. for each zip code list its corresponding location(s))

# create a database in current directory called temps.db
# conn = sqlite3.connect("temps.db")
# data = pd.read_sql_query(
#     """""",
#     conn
#     )

TypeError: 'NoneType' object is not iterable

In [ ]:
# take the average cost of housing for each zip code 
# (and attempt to find patterns)

# 5. Data Preprocessing

# 6. Model 

# 7. Model Evaluation

# 8. Model Tuning

# 9. Visualization of Results

# 10. Conclusion